## GBRegressor Experiment with MLflow

In [16]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error
from mlflow.models.signature import infer_signature
import pandas as pd
from sklearn.feature_extraction import DictVectorizer

In [17]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [18]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-02.parquet')

In [19]:
len(df_train), len(df_val)

(46307, 44218)

In [20]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [21]:
df_train['trip_type'] = df_train['trip_type'].astype(str)

In [22]:
categorical = ['PU_DO', 'trip_type'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [23]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

Linear Regression:

In [24]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

6.154305820280958

GradientBoostingRegressor:

In [25]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-gbregressor")

with mlflow.start_run() as run:
    mlflow.set_tag("model", "GradientBoostingRegressor")
    
    params = {
        "n_estimators": 100,
        "max_depth": 5,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "min_samples_leaf": 10,
        "random_state": 42
    }
    mlflow.log_params(params)
    
    gbr = GradientBoostingRegressor(**params)
    gbr.fit(X_train, y_train)
    
    y_pred = gbr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    
    signature = infer_signature(X_val, y_pred)
    mlflow.sklearn.log_model(gbr, name="model", signature=signature)
    
    print(f"RMSE: {rmse:.4f}")
    print(f"run_id: {run.info.run_id}")

2026/03/09 12:33:30 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpbfx75akl/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 


RMSE: 5.5316
run_id: 31571bbb326249e3b9cfa0314f52b657


RandomForest with Hyperopt

In [27]:
import pickle, os

os.makedirs("models", exist_ok=True)

# Save preprocessor locally first
with open("models/preprocessor.pkl", "wb") as f:
    pickle.dump(dv, f)

In [30]:
from sklearn.ensemble import RandomForestRegressor
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

def objective_rf(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "RandomForestRegressor")
        mlflow.log_params(params)
        
        rf = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        
        signature = infer_signature(X_val, y_pred)
        mlflow.sklearn.log_model(rf, name="model", signature=signature)

        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
        mlflow.sklearn.log_model("RandomForestRegressor", name="model", signature=signature)
        
    return {'loss': rmse, 'status': STATUS_OK}

search_space_rf = {
    'n_estimators': scope.int(hp.quniform('n_estimators', 50, 300, 10)),
    'max_depth': scope.int(hp.quniform('max_depth', 5, 30, 1)),
    'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 20, 1)),
    'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 10, 1)),
    'max_features': hp.choice('max_features', ['sqrt', 'log2'])
}

best_rf = fmin(
    fn=objective_rf,
    space=search_space_rf,
    algo=tpe.suggest,
    max_evals=20,       # fewer than XGBoost — RF is slower to train
    trials=Trials()
)

  0%|          | 0/20 [00:00<?, ?trial/s, best loss=?]

2026/03/09 14:11:03 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpowlj6mfp/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 

2026/03/09 14:11:03 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!

2026/03/09 14:11:05 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmprrihnh6r/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 



  5%|▌         | 1/20 [00:08<02:42,  8.55s/trial, best loss: 8.64603478568906]

2026/03/09 14:11:09 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp3l5p0387/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 

2026/03/09 14:11:09 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!

2026/03/09 14:11:11 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpwcl5hd3r/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 



 10%|█         | 2/20 [00:14<02:02,  6.81s/trial, best loss: 8.593626413707364]

...

 90%|█████████ | 18/20 [01:42<00:11,  5.56s/trial, best loss: 7.657335434382847]

2026/03/09 14:12:43 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpbk2bhv_5/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 

2026/03/09 14:12:43 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!

2026/03/09 14:12:44 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp_bxzqbpq/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 



 95%|█████████▌| 19/20 [01:47<00:05,  5.45s/trial, best loss: 7.657335434382847]

2026/03/09 14:12:48 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpt5v_cvez/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 

2026/03/09 14:12:48 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!

2026/03/09 14:12:50 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpz00ru6b2/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 



100%|██████████| 20/20 [01:53<00:00,  5.66s/trial, best loss: 7.657335434382847]
